# Stitch XCT TIFF Stacks (GPU-accelerated, dual-GPU)

Combine 3 sequential XCT volumes (~16GB each) into one stitched volume, using:

- **Cone-beam edge trimming** (top/bottom of every raw stack discarded before use)
- **Cone-beam edge trimming** with correct slice ordering verified: raw slice index 0 already
  represents the bottom of each stack's own scanned region (confirmed by manually-found overlap
  correspondences between stacks), so no flip/reversal is applied - only trimming
- **GPU-accelerated phase correlation** (PyTorch + CUDA) for registration - this is normally the
  slowest step, and benefits enormously from FFTs running on GPU instead of CPU
- **Both overlaps computed in parallel across your two GPUs** (overlap 1/2 on GPU 0, overlap 2/3
  on GPU 1, run concurrently) - since these are independent, we can use both cards at once
- Falls back automatically to CPU (skimage) if CUDA/PyTorch isn't available, so this notebook
  still runs correctly on a machine without GPUs, just slower

**Requirements:** `pip install torch tifffile numpy scipy scikit-image matplotlib`
(install the CUDA build of PyTorch appropriate for your system - see https://pytorch.org/get-started/locally/)

With 64GB RAM, streaming/memmap I/O (rather than full in-RAM volumes) is still used for the actual
read/write of the ~16GB stacks - this isn't a memory-saving compromise so much as it's simply
efficient and avoids holding ~48GB+ of stack data in memory unnecessarily. The GPU acceleration here
targets the *compute* bottleneck (registration), which is where your new hardware helps most.


In [ ]:
import numpy as np
import tifffile
import matplotlib.pyplot as plt
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from scipy.ndimage import shift as nd_shift

try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False

if TORCH_AVAILABLE:
    n_gpus = torch.cuda.device_count()
    print(f"PyTorch found. CUDA available: {torch.cuda.is_available()}. GPU count: {n_gpus}")
    for i in range(n_gpus):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    n_gpus = 0
    print("PyTorch not found - will fall back to CPU (skimage) registration. "
          "Install torch with CUDA support to use your GPUs.")

## 0b. Verify each GPU actually works

Some PyTorch/CUDA installs detect a GPU (`torch.cuda.is_available()` returns `True`) but don't have kernels compiled for that GPU's compute capability, which causes an `AcceleratorError: no kernel image is available for execution on the device` the first time you try to actually run something on it. This cell runs a tiny real computation on each detected GPU up front and marks it unusable (falling back to CPU) if it fails, instead of crashing deep inside the stitching run.

In [ ]:
def gpu_is_usable(device):
    if not TORCH_AVAILABLE:
        return False
    try:
        x = torch.ones((4, 4), device=device)
        _ = torch.fft.fftn(x)  # exercises the same op family used in registration
        return True
    except Exception as e:
        print(f"GPU {device} failed a functionality test and will not be used: {e}")
        return False


usable_devices = []
if TORCH_AVAILABLE:
    for i in range(n_gpus):
        dev = f"cuda:{i}"
        if gpu_is_usable(dev):
            usable_devices.append(dev)
            print(f"{dev}: OK")
        else:
            print(f"{dev}: NOT usable (likely a PyTorch/CUDA build mismatch - "
                  f"reinstall PyTorch matching your driver's CUDA version)")

print(f"\nUsable GPUs: {usable_devices if usable_devices else 'none - will use CPU'}")

## 0a. Reset GPU memory (run this if you've re-run cells after a previous OOM/crash)

If a prior attempt in this same kernel session hit an out-of-memory error or was interrupted, PyTorch's caching allocator can leave memory reserved on the GPU even though the tensors that used it are gone. This cell clears what it can. **Note:** this only helps for memory *this* kernel process is holding - if `nvidia-smi` shows a large chunk still in use immediately after running this, that's likely a different process (another notebook, another kernel) holding it, and the only real fix is stopping that other process or restarting this kernel entirely.

In [ ]:
import gc

if TORCH_AVAILABLE:
    for i in range(torch.cuda.device_count()):
        with torch.cuda.device(i):
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    gc.collect()
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f"cuda:{i} - free: {free / 1e9:.2f} GB / total: {total / 1e9:.2f} GB")
else:
    print("PyTorch not available - nothing to reset.")

## 1. Set your paths, trim, flip, and overlap estimates

In [ ]:
# --- EDIT THESE ---
STACK_1 = Path("/path/to/scan1_stack.tif")   # bottom
STACK_2 = Path("/path/to/scan2_stack.tif")   # middle
STACK_3 = Path("/path/to/scan3_stack.tif")   # top

OUTPUT_FILE = Path("/path/to/stitched_volume.tif")

TRIM_EDGE = 200  # unusable cone-beam-artefact slices at each end of every raw stack

# CORRECTED: raw slice index 0 already represents the BOTTOM of each stack's own scanned
# region (confirmed both by the original description and by the overlap correspondences
# found manually - see markdown above). Raw index therefore already increases with physical
# height in every stack, which is exactly the order needed for correct bottom->middle->top
# concatenation. No reversal is needed - flip is False for all three stacks.
FLIP_STACK_1 = False
FLIP_STACK_2 = False
FLIP_STACK_3 = False

# Overlap estimates refined from manually-identified corresponding slices (0-indexed, raw stacks):
#   bottom raw ~1800  <-> middle raw ~300   (usable-index offset ~1500)
#   middle raw ~1800  <-> top raw ~235      (usable-index offset ~1565)
# Converted to geometric overlap size in the trimmed "usable" coordinate system:
#   overlap_1_2 = (usable_length - offset_12) = (2028 - 2*200) - 1500 = 128
#   overlap_2_3 = (usable_length - offset_23) = (2028 - 2*200) - 1565 = 63
# These are still eyeballed estimates ("a range" per your own inspection), so margins stay
# generous rather than tight.
APPROX_OVERLAP_1_2 = 128
APPROX_OVERLAP_2_3 = 63
SEARCH_MARGIN_1_2 = 60   # search roughly 68-188 slices for overlap 1/2
SEARCH_MARGIN_2_3 = 40   # search roughly 23-103 slices for overlap 2/3

# With 64GB RAM and a fast workstation, larger chunks are fine and speed up I/O.
CHUNK_SIZE = 150

# Registration only needs to estimate a translation (the stage moved rigidly between scans),
# so it doesn't need the full frame - a central crop keeps GPU memory low regardless of how
# many candidate overlap sizes are searched. Blending/output still use full-resolution data;
# only the *registration search* is done on this smaller crop.
ROI_SIZE = 512  # pixels, centered crop in Y and X used for registration only

FEATHER = True

# Device assignment: auto-picks from GPUs that passed the functionality test in step 0b.
# Falls back to CPU for any device that isn't usable. You can override manually if you want
# a specific assignment.
if len(usable_devices) >= 2:
    DEVICE_1_2 = usable_devices[0]
    DEVICE_2_3 = usable_devices[1]
elif len(usable_devices) == 1:
    DEVICE_1_2 = usable_devices[0]
    DEVICE_2_3 = usable_devices[0]
else:
    DEVICE_1_2 = "cpu"
    DEVICE_2_3 = "cpu"

print(f"Overlap 1/2 will register on: {DEVICE_1_2}")
print(f"Overlap 2/3 will register on: {DEVICE_2_3}")
# -------------------

## 2. Open stacks and build "usable" views

Memory-mapped and lazily trimmed of cone-beam edges. `flip` is kept as a per-stack option in `usable_view()` for flexibility, but is set to False here: raw slice 0 already represents the bottom of each stack's own region (increasing raw index = increasing physical height, in every stack), so trimming alone already gives the correct bottom-to-top order needed for concatenation. bottom of each stack's region.

In [ ]:
def open_memmap(path):
    vol = tifffile.memmap(path, mode="r")
    print(f"{path.name}: raw shape={vol.shape} dtype={vol.dtype}")
    return vol


def usable_view(mm, trim, flip):
    trimmed = mm[trim: mm.shape[0] - trim]
    return trimmed[::-1] if flip else trimmed


mm1_raw = open_memmap(STACK_1)
mm2_raw = open_memmap(STACK_2)
mm3_raw = open_memmap(STACK_3)

assert mm1_raw.shape[1:] == mm2_raw.shape[1:] == mm3_raw.shape[1:], (
    "In-plane (X/Y) dimensions differ between stacks - check reconstructions used matching settings."
)

mm1 = usable_view(mm1_raw, TRIM_EDGE, FLIP_STACK_1)
mm2 = usable_view(mm2_raw, TRIM_EDGE, FLIP_STACK_2)
mm3 = usable_view(mm3_raw, TRIM_EDGE, FLIP_STACK_3)

print(f"\nUsable shape - stack1: {mm1.shape}, stack2: {mm2.shape}, stack3: {mm3.shape}")

## 3. Visual sanity check - CONFIRM BEFORE PROCEEDING

Same check as before: usable slice 0 should be the bottom of each stack's region (clean, no cone-beam artefact), usable last slice should be the top (about to overlap the next stack up). Fix `FLIP_STACK_n` and re-run from step 2 if a stack looks backwards.

In [ ]:
def show_check(mm_raw, mm_usable, name):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(np.asarray(mm_raw[0]), cmap="gray")
    axes[0].set_title(f"{name}: raw slice 0\n(expect cone-beam artefact)")
    axes[1].imshow(np.asarray(mm_usable[0]), cmap="gray")
    axes[1].set_title(f"{name}: usable slice 0\n(expect: bottom of region)")
    axes[2].imshow(np.asarray(mm_usable[-1]), cmap="gray")
    axes[2].set_title(f"{name}: usable last slice\n(expect: top of region)")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_check(mm1_raw, mm1, "Stack 1 (bottom)")
show_check(mm2_raw, mm2, "Stack 2 (middle)")
show_check(mm3_raw, mm3, "Stack 3 (top)")

## 4. GPU-accelerated phase correlation

Implements FFT-based phase correlation directly in PyTorch so it runs on GPU: computes the normalized cross-power spectrum via `torch.fft`, finds the integer-pixel peak, then refines to sub-pixel accuracy with a small parabolic fit around the peak in each axis. Falls back to `skimage.registration.phase_cross_correlation` on CPU automatically if CUDA/PyTorch isn't available.

Note: this sub-pixel refinement is a simpler parabolic estimate rather than skimage's upsampled-DFT method - it's a reasonable approximation given the feather-blend already smooths over small residual misalignment, but if you want maximum sub-pixel precision, the CPU fallback path used in the earlier version of this notebook (`skimage.registration.phase_cross_correlation` with `upsample_factor=4`) is more rigorous, just slower.

**Memory note:** registration only needs to estimate a translation (the stage moved rigidly between scans), so it's run on a central `ROI_SIZE x ROI_SIZE` crop of each slice rather than the full frame - full-resolution FFTs on every candidate overlap size would need far more GPU memory than most cards have. The crop's shift estimate applies directly to the full-resolution data (cropping only changes the window position, not pixel scale), and blending/output still use the full-resolution blocks - only the registration search itself is cropped.

In [ ]:
def _parabolic_subpixel(corr, peak_idx, axis_sizes):
    """Refine an integer peak index to sub-pixel precision per axis using a 3-point parabolic fit."""
    refined = list(peak_idx)
    for ax in range(len(peak_idx)):
        n = axis_sizes[ax]
        p = peak_idx[ax]
        p_prev = (p - 1) % n
        p_next = (p + 1) % n

        idx_prev = list(peak_idx); idx_prev[ax] = p_prev
        idx_next = list(peak_idx); idx_next[ax] = p_next

        y0 = corr[tuple(idx_prev)]
        y1 = corr[tuple(peak_idx)]
        y2 = corr[tuple(idx_next)]

        denom = (y0 - 2 * y1 + y2)
        if abs(denom) > 1e-8:
            delta = 0.5 * (y0 - y2) / denom
            delta = max(-1.0, min(1.0, float(delta)))  # keep the correction small/sane
        else:
            delta = 0.0
        refined[ax] = p + delta
    return refined


def gpu_phase_cross_correlation(block_a_np, block_b_np, device):
    """
    Returns (shift, error) where shift is (z, y, x) offset to apply to block_b to align with block_a,
    and error is a relative registration-quality score (lower = better, comparable across candidates
    within this run - not intended to match skimage's error scale exactly).
    """
    a = torch.from_numpy(block_a_np.astype(np.float32)).to(device)
    b = torch.from_numpy(block_b_np.astype(np.float32)).to(device)

    fa = torch.fft.fftn(a)
    fb = torch.fft.fftn(b)
    cross_power = fa * torch.conj(fb)
    cross_power = cross_power / (torch.abs(cross_power) + 1e-8)
    corr = torch.fft.ifftn(cross_power).real

    flat_idx = int(torch.argmax(corr).item())
    shape = tuple(corr.shape)
    peak_idx = list(np.unravel_index(flat_idx, shape))

    peak_val = float(corr[tuple(peak_idx)].item())
    error = 1.0 - peak_val  # corr is normalized, so peak close to 1.0 = good match

    refined_idx = _parabolic_subpixel(corr, peak_idx, shape)

    shifts = []
    for ax, s in enumerate(refined_idx):
        n = shape[ax]
        shift_val = s if s < n / 2 else s - n
        shifts.append(shift_val)

    del a, b, fa, fb, cross_power, corr
    if device != "cpu":
        torch.cuda.empty_cache()
    return np.array(shifts, dtype=float), error


def central_crop(block, roi_size):
    """Crop a centered (roi_size x roi_size) region out of each slice's Y/X plane.
    Registration only needs to estimate a translation, so a representative crop is
    sufficient and keeps GPU memory low regardless of full-frame resolution."""
    _, h, w = block.shape
    roi_h = min(roi_size, h)
    roi_w = min(roi_size, w)
    y0 = (h - roi_h) // 2
    x0 = (w - roi_w) // 2
    return block[:, y0:y0 + roi_h, x0:x0 + roi_w]


def cpu_phase_cross_correlation(block_a_np, block_b_np):
    from skimage.registration import phase_cross_correlation
    shift_est, error, _ = phase_cross_correlation(block_a_np, block_b_np, upsample_factor=4)
    return shift_est, error


def registered_shift(block_a_np, block_b_np, device):
    if TORCH_AVAILABLE and device != "cpu":
        return gpu_phase_cross_correlation(block_a_np, block_b_np, device)
    return cpu_phase_cross_correlation(block_a_np, block_b_np)

## 5. Overlap search + blending (GPU-aware)

In [ ]:
def find_best_overlap(mm_a, mm_b, approx_overlap, margin, device, roi_size=ROI_SIZE, min_overlap=10):
    max_n = approx_overlap + margin
    max_n = min(max_n, mm_a.shape[0], mm_b.shape[0])

    # Full-resolution blocks are read here (cheap - just a disk read into numpy, no GPU memory
    # involved) and kept around for later blending, but registration itself runs on a small
    # central crop of these to keep every GPU call small regardless of frame size or how many
    # candidate n's are searched.
    block_a_full = np.asarray(mm_a[-max_n:])
    block_b_full = np.asarray(mm_b[:max_n])

    block_a_crop_full = central_crop(block_a_full, roi_size)
    block_b_crop_full = central_crop(block_b_full, roi_size)

    candidates = range(max(min_overlap, approx_overlap - margin), max_n + 1)
    best = None

    for n in candidates:
        block_a_crop = block_a_crop_full[-n:]
        block_b_crop = block_b_crop_full[:n]
        shift_est, error = registered_shift(block_a_crop, block_b_crop, device)
        if best is None or error < best[2]:
            best = (n, shift_est, error)

    n, shift_est, error = best
    print(f"[{device}] Best overlap: {n} slices (ROI {roi_size}x{roi_size}), "
          f"shift(z,y,x)={shift_est}, error={error:.4f}")
    return n, shift_est, error, block_a_full, block_b_full


def build_histogram_lut(source_block, reference_block):
    """
    Builds a lookup table mapping source_block's gray values onto reference_block's distribution,
    via cumulative-histogram (quantile) matching - the same idea as
    skimage.exposure.match_histograms, but exposed as a reusable table (lut[value] -> corrected
    value) so it can be applied to an entire stack, not just the small block it was estimated from.
    """
    dtype = source_block.dtype
    if not np.issubdtype(dtype, np.integer):
        raise ValueError(
            f"build_histogram_lut expects integer CT data (got {dtype}) - "
            "adjust this function if your reconstructions are float."
        )

    src_values, src_counts = np.unique(source_block.ravel(), return_counts=True)
    ref_values, ref_counts = np.unique(reference_block.ravel(), return_counts=True)

    src_quantiles = np.cumsum(src_counts).astype(np.float64) / source_block.size
    ref_quantiles = np.cumsum(ref_counts).astype(np.float64) / reference_block.size

    interp_ref_values = np.interp(src_quantiles, ref_quantiles, ref_values)

    max_val = np.iinfo(dtype).max
    full_range = np.arange(max_val + 1)
    lut = np.interp(
        full_range, src_values, interp_ref_values,
        left=interp_ref_values[0], right=interp_ref_values[-1],
    )
    return np.clip(np.round(lut), 0, max_val).astype(dtype)


def apply_lut(block, lut):
    return block if lut is None else lut[block]


def feather_blend(block_a, block_b):
    n = block_a.shape[0]
    weights = np.linspace(1, 0, n).reshape(-1, 1, 1)
    blended = block_a.astype(np.float32) * weights + block_b.astype(np.float32) * (1 - weights)
    return blended.astype(block_a.dtype)


def compute_overlap_blend(mm_a, mm_b, approx_overlap, margin, device, feather=True, label=""):
    print(f"Finding overlap ({label}) on {device}...")
    n, shift_est, error, block_a_full, block_b_full = find_best_overlap(
        mm_a, mm_b, approx_overlap, margin, device
    )

    block_a = block_a_full[-n:]
    block_b = block_b_full[:n]

    # Sub-pixel shift correction stays on CPU (scipy) - block is small, not worth the data-transfer
    # overhead of doing this step on GPU too.
    block_b_aligned = nd_shift(block_b.astype(np.float32), shift=shift_est, order=1, mode="nearest")
    block_b_aligned = block_b_aligned.astype(block_b.dtype)

    # Build the brightness-correction LUT from this overlap (source=block_b, reference=block_a).
    # Returned alongside the blend so the SAME correction can be applied to the rest of that
    # stack (see step 7/8) - not just this overlap band.
    lut = build_histogram_lut(block_b_aligned, block_a)
    block_b_corrected = apply_lut(block_b_aligned, lut)

    blended = feather_blend(block_a, block_b_corrected) if feather else block_a

    del block_a_full, block_b_full
    return n, blended, lut

## 6. Run both overlaps in parallel across your two GPUs

These two registrations are independent, so they're launched concurrently via a thread pool - one targets `DEVICE_1_2`, the other `DEVICE_2_3`. If you have 2 GPUs, this genuinely runs both registrations at the same time on separate cards.

In [ ]:
with ThreadPoolExecutor(max_workers=2) as executor:
    future_12 = executor.submit(
        compute_overlap_blend, mm1, mm2, APPROX_OVERLAP_1_2, SEARCH_MARGIN_1_2, DEVICE_1_2,
        feather=FEATHER, label="1/2"
    )
    future_23 = executor.submit(
        compute_overlap_blend, mm2, mm3, APPROX_OVERLAP_2_3, SEARCH_MARGIN_2_3, DEVICE_2_3,
        feather=FEATHER, label="2/3"
    )
    n_12, blended_12, lut_middle = future_12.result()
    n_23, blended_23, lut_top_raw = future_23.result()

# lut_middle maps middle's raw gray values onto bottom's scale (bottom = reference, unchanged).
# lut_top_raw maps top's raw gray values onto (raw) middle's scale - but we want top expressed
# in bottom's scale too, so chain the two corrections: for each possible top-raw value, look up
# what it maps to in middle's scale, then look up what THAT maps to in bottom's scale.
composed_lut_top = lut_middle[lut_top_raw]

# blended_23 currently mixes raw-middle-scale data with top-corrected-to-middle-scale data (since
# it was built using middle as the local reference) - apply lut_middle to bring it onto bottom's
# scale too, consistent with the rest of the volume.
blended_23 = apply_lut(blended_23, lut_middle)

print(f"\noverlap 1/2: {n_12} slices | overlap 2/3: {n_23} slices")

## 7. Streaming write helpers

Both helpers now accept an optional `lut` - when given, it's applied to every chunk/block before writing, so the same brightness correction estimated from the overlap gets applied across the entire stack, not just the seam.

In [ ]:
def stream_copy(writer, mm_vol, start, end, chunk_size=CHUNK_SIZE, label="", lut=None):
    total = end - start
    if total <= 0:
        return
    for i in range(start, end, chunk_size):
        j = min(i + chunk_size, end)
        block = np.asarray(mm_vol[i:j])
        block = apply_lut(block, lut)
        for s in range(block.shape[0]):
            writer.write(block[s], contiguous=True)
        done = j - start
        print(f"  {label}: wrote {done}/{total} slices", end="\r")
    print()


def stream_write_block(writer, block, lut=None):
    block = apply_lut(block, lut)
    for s in range(block.shape[0]):
        writer.write(block[s], contiguous=True)

## 8. Stream the stitched volume to disk

`mm1` (bottom) is the brightness reference, written unchanged. `mm2` (middle) has `lut_middle` applied throughout. `mm3` (top) has `composed_lut_top` applied throughout, so it lands on the same scale as bottom via the middle-to-bottom correction. The two blended overlap blocks were already brought onto bottom's scale in steps 5/6, so they're written as-is here.

In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

len1, len2, len3 = mm1.shape[0], mm2.shape[0], mm3.shape[0]

with tifffile.TiffWriter(OUTPUT_FILE, bigtiff=True) as writer:
    stream_copy(writer, mm1, 0, len1 - n_12, label="stack 1 (head)")            # bottom: reference, no LUT
    stream_write_block(writer, blended_12)                                      # already bottom-scale
    stream_copy(writer, mm2, n_12, len2 - n_23, label="stack 2 (middle)", lut=lut_middle)
    stream_write_block(writer, blended_23)                                      # already bottom-scale
    stream_copy(writer, mm3, n_23, len3, label="stack 3 (tail)", lut=composed_lut_top)

final_len = (len1 - n_12) + n_12 + (len2 - n_12 - n_23) + n_23 + (len3 - n_23)
print(f"\nDone. Stitched volume written to {OUTPUT_FILE}")
print(f"Final slice count: {final_len}")

## 9. Sanity check the output

In [ ]:
stitched = tifffile.memmap(OUTPUT_FILE, mode="r")
print("Stitched volume shape:", stitched.shape, "dtype:", stitched.dtype)

seam_1 = len1 - n_12
seam_2 = seam_1 + n_12 + (len2 - n_12 - n_23)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, idx, title in zip(
    axes,
    [max(0, seam_1 - 30), seam_1, min(stitched.shape[0] - 1, seam_1 + 30)],
    ["before seam 1", "~seam 1", "after seam 1"],
):
    ax.imshow(np.asarray(stitched[idx]), cmap="gray")
    ax.set_title(f"{title} (slice {idx})")
    ax.axis("off")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, idx, title in zip(
    axes,
    [max(0, seam_2 - 30), seam_2, min(stitched.shape[0] - 1, seam_2 + 30)],
    ["before seam 2", "~seam 2", "after seam 2"],
):
    ax.imshow(np.asarray(stitched[idx]), cmap="gray")
    ax.set_title(f"{title} (slice {idx})")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Notes / troubleshooting

- **PyTorch/CUDA not detected:** check `torch.cuda.is_available()` in the setup cell output. If
  `False`, verify your CUDA driver + PyTorch build match (`pip install torch --index-url
  https://download.pytorch.org/whl/cu121` or similar, matching your CUDA version). The notebook
  will still run correctly on CPU via the automatic fallback, just slower.
- **Only one GPU detected but you have two:** check `nvidia-smi` in a terminal to confirm both are
  visible to the OS/driver; a container or environment may need `--gpus all` or equivalent to expose
  both.
- **Registration error is much higher/lower than expected in the GPU path vs CPU path:** the GPU
  path's `error` metric (`1 - peak_val`) is not on the same numeric scale as skimage's CPU error -
  they're only meant to be compared *within* the same path when picking the best candidate `n`, not
  against each other across paths.
- **Flip direction wrong / cone-beam trim insufficient:** same as before - check step 3's thumbnails
  before running the full stitch.
- **Disk space:** still need room for all three ~16GB raw stacks plus the full stitched output at
  once (roughly `3 x 16GB` minus overlap and trim).
- **Want maximum registration precision over speed:** set `DEVICE_1_2 = "cpu"` and
  `DEVICE_2_3 = "cpu"` manually in step 1 to force the skimage upsampled-DFT path instead of the
  GPU parabolic-refinement path.
- **`OutOfMemoryError` on the GPU during registration:** lower `ROI_SIZE` (e.g. to 256) - this
  directly controls how much data each FFT call needs to hold in VRAM, independent of your full
  frame resolution. If you still hit OOM at small ROI sizes, something else is likely holding GPU
  memory (e.g. another process) - check with `nvidia-smi` in a terminal.
- **Still see a brightness jump between stacks:** check that `build_histogram_lut`'s source/reference
  blocks actually contain representative material (not mostly air/background) - a LUT built from an
  unrepresentative overlap region won't generalize well to the rest of the stack. You can inspect
  `lut_middle`/`composed_lut_top` directly (e.g. `plt.plot(lut_middle)`) - a sensible correction
  should look like a smooth, mostly-monotonic curve, not something wildly non-monotonic or with a
  cliff in it.
- **Registration seems less accurate after the ROI crop change:** increase `ROI_SIZE` - a bigger
  crop gives phase correlation more structure to lock onto, at the cost of more GPU memory per
  call. 512 is a reasonable default for most CT cross-sections; textureless/very uniform regions
  may need a larger crop (or a crop repositioned away from the sample's center if that's mostly
  air).


## 10. Time-lapse viewer - play through the whole stitched volume

An interactive player (play button + scrub slider) for reviewing the entire stitched volume, not
just the seam regions. Reads slices from the memmap on demand, so it stays lightweight even on a
huge stack.

**Requirements:** `pip install ipywidgets pillow` (Pillow is usually already installed alongside
matplotlib/scikit-image).

Two settings control playback speed and smoothness:
- `SLICE_STEP` - view every Nth slice rather than all of them. For a quick full-volume check,
  skipping slices is fine - you're looking for gross problems (seams, dropped/duplicated frames,
  misalignment, artefacts), not inspecting every single slice in detail.
- `DISPLAY_DOWNSAMPLE` - shrinks each frame's in-plane resolution before display, so rendering
  keeps up with playback speed. This only affects the viewer, not your actual data.

Brightness scaling (`vmin`/`vmax`) is computed once from a sample of frames across the volume, so
image brightness stays consistent as you play through rather than auto-scaling per slice (which
would look like flickering).


In [ ]:
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
import io

SLICE_STEP = 5            # view every Nth slice
DISPLAY_DOWNSAMPLE = 4     # in-plane downsample factor for the viewer only
PLAY_INTERVAL_MS = 80      # milliseconds between frames during playback

indices = list(range(0, stitched.shape[0], SLICE_STEP))

# Sample a spread of frames across the volume to fix a consistent brightness range
sample_idxs = indices[::max(1, len(indices) // 20)]
sample_stack = np.stack([
    np.asarray(stitched[i])[::DISPLAY_DOWNSAMPLE, ::DISPLAY_DOWNSAMPLE] for i in sample_idxs
])
vmin, vmax = np.percentile(sample_stack, [0.5, 99.5])
del sample_stack


def to_png_bytes(slice_idx):
    frame = np.asarray(stitched[slice_idx])[::DISPLAY_DOWNSAMPLE, ::DISPLAY_DOWNSAMPLE].astype(np.float32)
    frame = np.clip((frame - vmin) / (vmax - vmin + 1e-8) * 255, 0, 255).astype(np.uint8)
    buf = io.BytesIO()
    Image.fromarray(frame).save(buf, format="PNG")
    return buf.getvalue()


image_widget = widgets.Image(value=to_png_bytes(indices[0]), format="png", width=500)
label_widget = widgets.Label(value=f"slice {indices[0]}/{stitched.shape[0] - 1}")


def on_change(change):
    slice_idx = indices[change["new"]]
    image_widget.value = to_png_bytes(slice_idx)
    label_widget.value = f"slice {slice_idx}/{stitched.shape[0] - 1}"


play = widgets.Play(min=0, max=len(indices) - 1, step=1, interval=PLAY_INTERVAL_MS, description="Play")
slider = widgets.IntSlider(min=0, max=len(indices) - 1, step=1, description="frame")
widgets.jslink((play, "value"), (slider, "value"))
slider.observe(on_change, names="value")

display(widgets.VBox([widgets.HBox([play, slider]), label_widget, image_widget]))

**Tips:**
- Drag the slider to scrub manually, or hit the play button (▶) for continuous auto-playback.
- If playback looks choppy, increase `SLICE_STEP` or `DISPLAY_DOWNSAMPLE` and re-run the cell.
- If you want a shareable video file (e.g. an actual `.mp4`/`.gif` rather than an in-notebook
  viewer), that's a quick addition using `imageio` - ask if you'd like that cell added too.
